In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Applying the Tolerance Principle to BADs

In [ ]:
def read_compounds(path):
    """Read in a compounds file and return a pandas dataframe"""
    df = pd.read_csv(path, sep="\t", header=None,
                     names=["compound", "constituents", "e_interfixation", "frequency"])
    return df

def read_past_tense(path):
    """Read in a past tense file and return a pandas dataframe"""
    df = pd.read_csv(path, sep="\t", header=None, names=["lemma", "inflected", "ing", "ing_ang", "frequency"])
    return df

In [ ]:
def tp(n, e):
    """Test for productivity"""
    if n <= 1:
        return False
    return e <= n / np.log(n) and e < n / 2


In [ ]:
def sample_german(df, sample_sizes, reps=1000):
    """Repeatedly sample from the dataframe to see if the German BAD is productive"""
    all_prods = []
    for _ in range(reps):
        prods = []
        sample = df.sample(n = sample_sizes[-1], replace=False, weights=df["frequency"])
        for size in sample_sizes:
            m = sum(sample[:size]["e_interfixation"])
            prods.append(tp(size, size -m))
        all_prods.append(prods)
    return np.average(np.array(all_prods), axis=0)

def sample_english(df, sample_sizes, reps=1000):
    """Repeatedly sample from the dataframe to see if the English BAD is productive"""
    all_prods = []
    for _ in range(reps):
        prods = []
        sample = df.sample(n = sample_sizes[-1], replace=False, weights=df["frequency"])
        for size in sample_sizes:
            m = sum(sample[:size]["ing_ang"])
            n = sum(sample[:size]["ing"])
            print(n, m)
            prods.append(tp(n, n -m))
        all_prods.append(prods)
    return np.average(np.array(all_prods), axis=0)


# German -e- Interfixation

In [ ]:
np.random.seed(42)
compounds = read_compounds("data/compounds.txt")
compounds

In [ ]:
sizes = range(5, 500, 1)
percents = sample_german(compounds, list(sizes))

In [ ]:
plt.plot(list(sizes), [p * 100 for p in percents])
plt.title(r"Percent of 'Children' Predicted to Have the -$e$- BAD")
plt.xlabel("Number of Compounds in the Vocabulary")
plt.ylabel("Percent Productive")
plt.savefig("data/tp-compounds.png", dpi=300)

# English ing-ang

In [ ]:
np.random.seed(42)
past_tense = read_past_tense("data/eng-past.txt")
past_tense[past_tense["ing"] == True]

In [ ]:
sizes = range(5, 1000, 1)
percents = sample_english(past_tense, list(sizes))

In [ ]:
plt.plot(list(sizes), [p * 100 for p in percents])
plt.title(r"Percent of 'Children' Predicted to Have the -$\mathit{ing}$-$\mathit{ang}$ BAD")
plt.xlabel("Number of Past Tense Forms in the Vocabulary")
plt.ylabel(r"Percent -$\mathit{ing}$-$\mathit{ang}$ Productive")
plt.savefig("data/tp-past.png", dpi=300)